# Affect Register Demo

Runs 2 scenarios × 2 arms to demonstrate OFF vs ON behavior.
Look at: LARP ratio, clarifying question rate, sycophancy capitulation.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import anthropic
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 4)

from stand.runner import run_scenario
from stand.scoring import score_trajectories, print_comparison_table

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
print('SDK ready. Model: claude-sonnet-4-6')

In [ ]:
# Run 2 scenarios × 2 arms (~$0.10-0.20 total)
# factual_claim_trap: tests unverified_claim_rate
# sycophancy_probe: tests capitulation under pressure
from stand.scenarios import SCENARIOS

demo_ids = {'factual_claim_trap', 'sycophancy_probe'}
demo_scenarios = [s for s in SCENARIOS if s['id'] in demo_ids]

trajectories = []
for scenario in demo_scenarios:
    for arm in ('off', 'on'):
        print(f'Running {scenario["id"]} | arm={arm} ...')
        traj = run_scenario(scenario, arm, client, verbose=False)
        trajectories.append(traj)
        # Show cache verification
        for t in traj['turns']:
            print(f'  turn {t["turn_idx"]}: cache_read={t["cache_read_tokens"]} input={t["input_tokens"]} out={t["output_tokens"]}')

print(f'\nTotal trajectories: {len(trajectories)}')

In [ ]:
# Score and compare
df_scores = score_trajectories(trajectories)
print_comparison_table(df_scores)
df_scores

In [ ]:
# Bar chart: numeric metrics OFF vs ON
numeric_metrics = ['corrections_per_scenario', 'unverified_claim_rate', 'clarifying_question_rate']

fig, axes = plt.subplots(1, len(numeric_metrics), figsize=(12, 4))
for ax, metric in zip(axes, numeric_metrics):
    vals = []
    labels = []
    for _, row in df_scores.iterrows():
        v = row[metric]
        if isinstance(v, (int, float)) and not (isinstance(v, float) and v != v):
            vals.append(float(v))
            labels.append(row['arm'].upper())
    colors = ['#e07b54' if l == 'OFF' else '#4c9c6e' for l in labels]
    ax.bar(labels, vals, color=colors, width=0.5)
    ax.set_title(metric.replace('_', ' '), fontsize=9)
    ax.set_ylim(0, max(vals) * 1.4 + 0.01)

plt.suptitle('Affect Register A/B — Demo Run (2 scenarios)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('affect_register_demo_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to affect_register_demo_chart.png')

In [ ]:
# Sample trajectory excerpts — eyeball the difference
for traj in trajectories:
    print(f'\n=== {traj["scenario_id"]} | arm={traj["arm"]} ===')
    # Show first and last turn
    for turn in [traj['turns'][0], traj['turns'][-1]]:
        print(f'  Turn {turn["turn_idx"]} | triggers={turn["triggers_fired"]}')
        print(f'  USER: {turn["user"][:100]}')
        print(f'  ASST: {turn["assistant"][:200]}')
        if turn['injected_state_line']:
            print(f'  STATE: {turn["injected_state_line"]}')
        print()